# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook guides you through loading, inspecting, and exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library. You'll use Croissant metadata—referencing all entities by their `@id`—to systematically explore and analyze the data.

### Dataset Source
The dataset source is defined via its Croissant schema URL.

In [ ]:
# Make sure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished}, Version: {metadata.version}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In Croissant, each entity is referenced by its `@id`. Here, we'll print the record sets and their fields, showing their respective `@id` values.

In [ ]:
# List record sets from the metadata
record_sets = metadata.recordSet
if not record_sets:
    # Try loading the schema directly as dictionary if recordSet is empty
    import requests
    schema = requests.get(croissant_url).json()
    if 'recordSet' in schema:
        record_sets = schema['recordSet']

if record_sets:
    print("Available record sets:")
    for rs in record_sets:
        rs_id = rs.get('@id') if isinstance(rs, dict) else rs
        rs_name = rs.get('name', rs_id) if isinstance(rs, dict) else rs_id
        print(f"- Record set @id: {rs_id}, name: {rs_name}")
        # List fields
        fields = rs.get('field', []) if isinstance(rs, dict) else []
        for fld in fields:
            fld_id = fld.get('@id') if isinstance(fld, dict) else fld
            fld_name = fld.get('name', fld_id) if isinstance(fld, dict) else fld_id
            print(f"  - Field @id: {fld_id}, name: {fld_name}")
else:
    print("No record sets found in metadata.")

# For this dataset, manually guess the main record set @id if needed
# Commonly, it's provided. For this example, let's use the actual recordSet @id if available.

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s discovered above.

We'll find the proper record set `@id` from the previous cell. If none found, we'll fetch the underlying schema and extract the root RecordSet.

In [ ]:
# Resolve available record set IDs
record_set_ids = []
if record_sets:
    for rs in record_sets:
        if isinstance(rs, dict):
            record_set_ids.append(rs['@id'])
        elif isinstance(rs, str):
            record_set_ids.append(rs)

# If not found, extract from schema
if not record_set_ids:
    # Try loading from Croissant schema
    import requests
    schema = requests.get(croissant_url).json()
    if 'recordSet' in schema:
        if isinstance(schema['recordSet'], list):
            for rs in schema['recordSet']:
                if isinstance(rs, dict) and '@id' in rs:
                    record_set_ids.append(rs['@id'])
                elif isinstance(rs, str):
                    record_set_ids.append(rs)

print("Record set @id(s):", record_set_ids)

# Load data from each record set
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded {len(df)} records from record set {rs_id}")

# Display column names for the first record set
if record_set_ids:
    first_rs_id = record_set_ids[0]
    print(f"Columns in record set {first_rs_id}:")
    print(dataframes[first_rs_id].columns.tolist())
    dataframes[first_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping the data by categories. We reference all columns using their `@id`.

In [ ]:
# We'll pick a numeric field for filtering and normalization
# For FAIR² Mental Health Survey, possible numeric fields may be GAD-7, PHQ-9, or PSQ scores.
# To demonstrate referencing by @id, list all columns and pick one
df = dataframes[first_rs_id]
print("All columns:", df.columns.tolist())

# Suppose GAD-7 score column has @id 'cr:gad7_score', adjust as needed
possible_numeric_fields = [col for col in df.columns if 'score' in col]
print("Possible numeric fields:", possible_numeric_fields)
if possible_numeric_fields:
    numeric_field_id = possible_numeric_fields[0]
else:
    # If no score field found, pick Age or another numeric field
    numeric_field_id = 'cr:age' if 'cr:age' in df.columns else df.select_dtypes('number').columns[0]

threshold = 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a categorical field, e.g., 'cr:gender' if present
possible_group_fields = [col for col in df.columns if 'gender' in col.lower() or 'marital' in col.lower() or 'education' in col.lower()]
if possible_group_fields:
    group_field_id = possible_group_fields[0]
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll use `matplotlib` to show histograms and boxplots of numeric fields, as well as group comparisons.

In [ ]:
# Histogram of the selected numeric field
plt.figure(figsize=(6, 4))
df[numeric_field_id].plot.hist(bins=20, color='skyblue', edgecolor='black')
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Frequency")
plt.show()

# Box plot grouped by group_field_id if available
if 'group_field_id' in locals():
    plt.figure(figsize=(8, 5))
    df.boxplot(column=numeric_field_id, by=group_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.suptitle('')
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

# Scatter plot between two scores if available
if len(possible_numeric_fields) > 1:
    plt.figure(figsize=(6, 4))
    plt.scatter(df[possible_numeric_fields[0]], df[possible_numeric_fields[1]], c='purple', alpha=0.5)
    plt.xlabel(possible_numeric_fields[0])
    plt.ylabel(possible_numeric_fields[1])
    plt.title(f"{possible_numeric_fields[0]} vs {possible_numeric_fields[1]}")
    plt.show()

## 6. Conclusion
This notebook demonstrated systematic, reproducible exploration of the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya, referencing all Croissant entities by their `@id`. We covered metadata review, record set inspection, DataFrame extraction, numeric field normalization, grouping, and basic visualizations.

Key findings:
- The dataset is rich in demographic and mental health survey data, suitable for quantitative analysis and public health research.
- Using Croissant and `mlcroissant`, entities and fields are uniquely referenced, supporting transparent and robust data workflows.
- Filtering and normalizing by score fields (e.g., GAD-7, PHQ-9) enables meaningful segmentation and comparison.
- Visualizations reveal distributions and group effects useful for statistical and community intervention studies.

For further analysis, consider cross-referencing summary statistics and exploring additional fields defined by their `@id` in follow-up notebooks.